# ModelEvaluator — Manual Validation Notebook

Runs the evaluator on real preprocessed EEG epochs and visualises the results.

**Decoder tasks:** one-vs-all across `object`, `scene`, and `feature` conditions.

**This notebook is for validation only — not part of the app.**

Expected runtime with `QUICK_MODE = True` (0–0.8 s crop, 3-fold CV): **2–5 minutes**.  
Set `QUICK_MODE = False` for the full −0.6–0.99 s window with 5-fold CV (~15–30 min).

In [ ]:
import sys
from pathlib import Path

# ── Locate repo root regardless of where Jupyter was launched ─────────────
# Searches upward from CWD for the directory that contains data/preprocessed/
_repo = next(
    (p.resolve() for p in [Path("."), Path(".."), Path("../.."), Path("../../..")]
     if (p / "data" / "preprocessed").is_dir()),
    None,
)
assert _repo is not None, (
    "Could not find repo root (expected a parent directory containing data/preprocessed/). "
    "Launch Jupyter from within the reactivation-decoder project tree."
)

# ── Locate online_decoder/src ─────────────────────────────────────────────
_src = next(
    (p.resolve() for p in [Path("src"), Path("../src"), Path("../../src")]
     if (p / "backend").is_dir()),
    None,
)
assert _src is not None, (
    "Could not find src/backend/. "
    "Launch Jupyter from within the online_decoder/ directory tree."
)
sys.path.insert(0, str(_src))
# ─────────────────────────────────────────────────────────────────────────

# ── Data source ───────────────────────────────────────────────────────────
# "preprocessed"  — load an existing preprocessed .fif directly (fast)
# "full_pipeline" — run OfflinePreprocessor from raw .vhdr (slow, ~5–10 min)
DATA_SOURCE = "preprocessed"

EPOCHS_PATH = _repo / "data/preprocessed/sub_001/sub_001_prepro-epo.fif"
RAW_DIR     = _repo / "data/raw/sub_001/"
CONFIG_PATH = _src.parent.parent / "experiment_config.yaml"   # online_decoder/experiment_config.yaml
OUTPUT_DIR  = _repo / "data_vault/sub_001/"
# ─────────────────────────────────────────────────────────────────────────

# Quick mode: crop to 0–0.8 s and use 3-fold CV for faster iteration.
# Set to False for the full window with 5-fold CV.
QUICK_MODE = True

import mne
import numpy as np
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt
import time

from backend.core.settings_manager import SettingsManager
from backend.offline_phase.preprocessor import OfflinePreprocessor
from backend.offline_phase.evaluator import ModelEvaluator

mne.set_log_level("WARNING")
print("Imports OK")
print(f"Repo root  : {_repo}")
print(f"src        : {_src}")
print(f"Data source: {DATA_SOURCE}")
print(f"Quick mode : {QUICK_MODE}")

---
## Load Epochs

Two paths depending on `DATA_SOURCE`:

- **`"preprocessed"`** — reads an existing `.fif` file produced by a previous run.  
- **`"full_pipeline"`** — runs `OfflinePreprocessor` from the raw `.vhdr`, pauses for ICA  
  review (suggested components are printed; adjust `EXCLUDE_COMPONENTS` in the next cell),  
  then finishes the pipeline and loads the resulting epochs.

In [ ]:
if DATA_SOURCE == "preprocessed":
    assert EPOCHS_PATH.exists(), (
        f"File not found: {EPOCHS_PATH.resolve()}\n"
        "Adjust EPOCHS_PATH in the config cell, or set DATA_SOURCE = 'full_pipeline'."
    )
    epochs = mne.read_epochs(EPOCHS_PATH, preload=True, verbose=False)
    preprocessor = None  # not needed downstream

elif DATA_SOURCE == "full_pipeline":
    assert RAW_DIR.exists(),     f"RAW_DIR not found: {RAW_DIR.resolve()}"
    assert CONFIG_PATH.exists(), f"CONFIG_PATH not found: {CONFIG_PATH.resolve()}"

    settings        = SettingsManager(CONFIG_PATH)
    prepro_params   = settings.get_preprocessing_params()
    event_mapping   = settings.get_event_mapping()   # {id: name}

    preprocessor = OfflinePreprocessor(
        data_dir=RAW_DIR,
        preprocessing_settings=prepro_params,
    )
    print(f"Subject    : {preprocessor.subject_id}")
    print(f"VHDR file  : {preprocessor.vhdr}")
    assert preprocessor.vhdr is not None, "No .vhdr found — check RAW_DIR"

    print("\nRunning Step 1 (filter → resample → bad channels → ICA)…")
    suggested = preprocessor.run_step1_prepare_ica()
    print(f"\nSuggested artifact components: {suggested}")
    print("→ Review the components, then set EXCLUDE_COMPONENTS in the next cell and run it.")

else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE!r}. Use 'preprocessed' or 'full_pipeline'.")

In [ ]:
# ── Full pipeline only: ICA review + Step 2 ──────────────────────────────
# Skip this cell when DATA_SOURCE = "preprocessed" (preprocessor is None).
if preprocessor is not None:
    # Start from auto-suggestions; add any components you identified above.
    EXCLUDE_COMPONENTS = list(suggested)
    # EXCLUDE_COMPONENTS += [0, 3]   # ← add indices after visual inspection

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Excluding components : {EXCLUDE_COMPONENTS}")
    print(f"Output dir           : {OUTPUT_DIR.resolve()}")

    preprocessor.run_step2_finish_pipeline(
        exclude_components=EXCLUDE_COMPONENTS,
        event_mapping=event_mapping,
        output_dir=OUTPUT_DIR,
    )
    epochs = preprocessor.epochs
    assert epochs is not None and len(epochs) > 0, "Step 2 produced no epochs"
    print(f"\n✓ Pipeline complete — {len(epochs)} epochs")
else:
    print("Skipped (DATA_SOURCE = 'preprocessed')")

In [ ]:
# Both paths converge here — crop if quick mode, then print info.
if QUICK_MODE:
    epochs = epochs.crop(tmin=0.0, tmax=0.8)

print(f"n_epochs   : {len(epochs)}")
print(f"n_channels : {len(epochs.ch_names)}")
print(f"n_times    : {len(epochs.times)}")
print(f"time range : {epochs.tmin:.2f} s  →  {epochs.tmax:.2f} s")
print(f"sfreq      : {epochs.info['sfreq']:.0f} Hz")
print("\nEvent counts:")
for name, code in sorted(epochs.event_id.items(), key=lambda x: x[1]):
    count = int((epochs.events[:, 2] == code).sum())
    print(f"  {name:14s} (id={code:3d}): {count} epochs")

assert len(epochs) > 0, "No epochs loaded"
assert epochs.preload,   "Epochs must be preloaded"
print("\n✓ Epochs ready")

---
## Define Decoder Tasks

One-vs-all classification for the three visual categories present in the localizer.

In [ ]:
CV_FOLDS = 3 if QUICK_MODE else 5

decoder_settings = {
    "model": "LDA",
    "params": {"solver": "lsqr", "shrinkage": "auto"},
    "scale_method": "standard",
    "cv": {"k": CV_FOLDS},
    "random_state": 42,
    "tasks": [
        {
            "name": "object vs scene+feature",
            "pos_labels": ["object"],
            "neg_labels": ["scene", "feature"],
        },
        {
            "name": "scene vs object+feature",
            "pos_labels": ["scene"],
            "neg_labels": ["object", "feature"],
        },
        {
            "name": "feature vs object+scene",
            "pos_labels": ["feature"],
            "neg_labels": ["object", "scene"],
        },
    ],
}

print(f"CV folds : {CV_FOLDS}")
print("Tasks    :")
for t in decoder_settings["tasks"]:
    print(f"  {t['name']}")
    print(f"    pos: {t['pos_labels']}  neg: {t['neg_labels']}")

# Validate labels exist in the loaded epochs
known_labels = set(epochs.event_id.keys())
for t in decoder_settings["tasks"]:
    for lbl in t["pos_labels"] + t["neg_labels"]:
        assert lbl in known_labels, f"Label '{lbl}' not in epochs.event_id: {known_labels}"

print("\n✓ All task labels found in epochs")

---
## Run Evaluation

`ModelEvaluator` runs one `GeneralizingEstimator` CV pass per task — one pass produces
both the TGM and the diagonal AUC curve.  

Quick mode (~80 timepoints, 3-fold): **2–5 minutes**.  
Full mode (~160 timepoints, 5-fold): **15–30 minutes**.

In [ ]:
print("Running ModelEvaluator.run_evaluation()...")
print(f"  Tasks    : {len(decoder_settings['tasks'])}")
print(f"  CV folds : {decoder_settings['cv']['k']}")
print(f"  n_times  : {len(epochs.times)}")
print()

t0 = time.time()
evaluator = ModelEvaluator(epochs, decoder_settings)
results = evaluator.run_evaluation()
elapsed = time.time() - t0

print(f"Done in {elapsed:.0f} s")

# ── Structural assertions ─────────────────────────────────────────────────
assert set(results.keys()) == {"times", "suggested_timepoint", "average_peak_auc", "tasks"}
assert len(results["tasks"]) == len(decoder_settings["tasks"])

n_times = len(epochs.times)
for task_name, task_data in results["tasks"].items():
    assert task_data["diagonal_auc"].shape == (n_times,),         f"{task_name}: diagonal shape wrong"
    assert task_data["tgm_matrix"].shape   == (n_times, n_times), f"{task_name}: TGM shape wrong"
    assert task_data["peak_auc"]           == task_data["diagonal_auc"].max()
    assert task_data["chance_level"]       == 0.5

print("\n✓ Result structure OK")

---
## Results

In [ ]:
# Summary table
print(f"{'Task':<30}  {'Peak AUC':>9}  {'Chance':>7}")
print("-" * 52)
for name, td in results["tasks"].items():
    print(f"{name:<30}  {td['peak_auc']:>9.4f}  {td['chance_level']:>7.2f}")

print()
print(f"Suggested timepoint : {results['suggested_timepoint']:.3f} s")
print(f"Average peak AUC    : {results['average_peak_auc']:.4f}")

In [ ]:
# Diagonal AUC — one line per decoder task
fig, ax = plt.subplots(figsize=(10, 4))

times = results["times"]
colors = plt.cm.tab10.colors

for i, (name, td) in enumerate(results["tasks"].items()):
    ax.plot(times, td["diagonal_auc"], label=name, color=colors[i], lw=1.8)

ax.axhline(0.5, color="gray", ls="--", lw=1, label="chance (0.5)")
ax.axvline(
    results["suggested_timepoint"],
    color="black", ls=":", lw=1.5,
    label=f"suggested t = {results['suggested_timepoint']:.3f} s",
)
ax.axvline(0.0, color="silver", ls="-", lw=0.8, alpha=0.6)

ax.set_xlabel("Time (s)")
ax.set_ylabel("AUC (ROC)")
ax.set_title(f"Diagonal Decoding Accuracy — sub_001  (CV = {decoder_settings['cv']['k']}-fold)")
ax.set_ylim(0.3, 1.0)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# TGM heatmaps — one subplot per decoder task
n_tasks = len(results["tasks"])
fig, axes = plt.subplots(1, n_tasks, figsize=(5 * n_tasks, 4.5))

if n_tasks == 1:
    axes = [axes]

for ax, (name, td) in zip(axes, results["tasks"].items()):
    t = results["times"]
    im = ax.imshow(
        td["tgm_matrix"],
        origin="lower",
        extent=[t[0], t[-1], t[0], t[-1]],
        aspect="auto",
        vmin=0.3, vmax=0.9,
        cmap="RdBu_r",
    )
    # Diagonal reference
    ax.plot([t[0], t[-1]], [t[0], t[-1]], "k--", lw=0.8, alpha=0.5)
    ax.axvline(0, color="white", lw=0.8, alpha=0.4)
    ax.axhline(0, color="white", lw=0.8, alpha=0.4)
    ax.axvline(results["suggested_timepoint"], color="yellow", lw=1.0, alpha=0.7)
    ax.set_xlabel("Test time (s)")
    ax.set_ylabel("Train time (s)")
    ax.set_title(name, fontsize=10)
    plt.colorbar(im, ax=ax, label="AUC")

plt.suptitle("Temporal Generalization Matrices — sub_001", y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

If all cells passed:

- **Diagonal AUC** should rise clearly above 0.5 in the post-stimulus window if the EEG signal is decodable.
- **TGM off-diagonal** spread indicates how far a code trained at one time generalises to other times.
- **Suggested timepoint** marks where the mean decoder performance peaks — the ModelTrainer will use this as the default training time.

**Next step:** pass `suggested_timepoint` and the cleaned epochs to `ModelTrainer` to produce the final `decoder_pipeline.joblib`.